# 13. Optimize a Correct Implementation

Optimization order: **correctness → benchmark → data pipeline → mixed precision → profiler → compile → scale hardware**.

## Automatic mixed precision (current PyTorch style)

Use `torch.autocast` and `torch.amp.GradScaler`. Benchmark accuracy and numerical stability; mixed precision is not universally safe for every model/loss.

In [ ]:
import torch
from torch import nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = nn.Conv2d(1, 1, 3, padding=1).to(device)
opt = torch.optim.Adam(model.parameters(), 1e-3)
loss_fn = nn.MSELoss()
x = torch.rand(4,1,64,64, device=device)
y = torch.rand_like(x)

if device.type == 'cuda':
    scaler = torch.amp.GradScaler('cuda')
    opt.zero_grad(set_to_none=True)
    with torch.autocast(device_type='cuda', dtype=torch.float16):
        loss = loss_fn(model(x), y)
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
else:
    opt.zero_grad(set_to_none=True)
    loss = loss_fn(model(x), y)
    loss.backward(); opt.step()
print(float(loss))

## Profile before guessing

`torch.profiler` can show expensive operators, CPU/GPU activity, input shapes, and memory behavior. Profile representative steps—not an entire week-long training run.

In [ ]:
from torch.profiler import profile, ProfilerActivity
activities = [ProfilerActivity.CPU]
if torch.cuda.is_available(): activities.append(ProfilerActivity.CUDA)
with profile(activities=activities, record_shapes=True) as prof:
    _ = model(x)
print(prof.key_averages().table(sort_by='self_cpu_time_total', row_limit=5))

## `torch.compile`

Compilation has startup cost and benefits depend on model/hardware. Benchmark eager vs compiled after the model is correct. Graph breaks can reduce optimization opportunities.

In [ ]:
if hasattr(torch, 'compile'):
    compiled_model = torch.compile(model)
    print('compiled model created; benchmark before deciding to keep it')

## Other high-value optimizations

- right-size image/patch dimensions;
- use efficient DataLoader workers/pinned memory when beneficial;
- avoid repeated CPU↔GPU copies;
- choose batch size based on memory/throughput, not habit;
- use gradient accumulation when memory requires it;
- save only checkpoints you need;
- do not move to multi-GPU until a single-GPU profile justifies it.